# Pipeline SF3D di Kaggle — dataset `scrapping grafkom`

Jalankan tahap A–E di Kaggle **GPU T4**. Input = Kaggle Dataset, output ke `/kaggle/working`.

**Sebelum mulai (panel kanan Kaggle):**
1. **Settings → Accelerator → GPU T4 x1**.
2. **Settings → Internet → ON** (perlu untuk pip + git + Hugging Face).
3. **Add Data →** cari & attach dataset **`scrapping grafkom`**.
4. **Add-ons → Secrets →** tambah secret `HF_TOKEN` (token Hugging Face, sudah accept license SF3D).

Struktur dataset (dari scraping): `dataset_artefak_indonesia/<id>_<nama>.jpg` + `met_dataset/`.

In [ ]:
#@title 1. Cek GPU + temukan path dataset otomatis
import glob, os
!nvidia-smi -L
roots = glob.glob('/kaggle/input/*')
print('Dataset ter-attach:', roots)
# cari folder foto artefak (nama bisa beda-tipis tergantung slug)
cands = glob.glob('/kaggle/input/**/dataset_artefak_indonesia', recursive=True)
INPUT = cands[0] if cands else (roots[0] if roots else '')
print('INPUT =', INPUT)
print('contoh isi:', os.listdir(INPUT)[:5] if INPUT else 'KOSONG - attach dataset dulu')

In [ ]:
#@title 2. Pasang SF3D (internet harus ON)
%cd /kaggle/working
!git clone -q https://github.com/Stability-AI/stable-fast-3d.git sf3d_repo
%cd sf3d_repo
!pip install -q -r requirements.txt
!pip install -q ./texture_baker ./uv_unwrapper
!pip install -q omegaconf trimesh rembg onnxruntime
%cd /kaggle/working

In [ ]:
#@title 3. Clone repo tim + login Hugging Face dari Kaggle Secret
REPO_URL = "https://github.com/eycoo/FP_GRAFKOM.git"  #@param {type:"string"}
import os, sys
if not os.path.exists('/kaggle/working/FP_GRAFKOM'):
    !git clone -q $REPO_URL /kaggle/working/FP_GRAFKOM
sys.path.insert(0, '/kaggle/working/sf3d_repo')
sys.path.insert(0, '/kaggle/working/FP_GRAFKOM/pipeline/src')

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))
print('HF login OK')

In [ ]:
#@title 4. Muat SF3D + config (output ke /kaggle/working/outputs)
from config import PipelineCfg
from inference import SF3DReconstructor
from preprocess import load_rembg_session

cfg = PipelineCfg.load('/kaggle/working/FP_GRAFKOM/pipeline/configs/default.yaml',
                       **{'export.out_dir': '/kaggle/working/outputs'})
reconstructor = SF3DReconstructor(cfg.model)
rembg_session = load_rembg_session()
print('device:', reconstructor.device)

In [ ]:
#@title 5. Inferensi batch (LIMIT dulu untuk uji cepat)
from run_pipeline import collect_images, process_one
from export_glb import write_manifest
from pathlib import Path

LIMIT = 5     #@param {type:"integer"}  # 0 = semua foto
TIER  = "high"  #@param ["high", "mid", "low"]
manifest = Path(cfg.export.out_dir) / 'manifest.jsonl'

images = collect_images(Path(INPUT))
if LIMIT: images = images[:LIMIT]
print(f'proses {len(images)} foto')
for i, img in enumerate(images, 1):
    try:
        rec = process_one(img, reconstructor, cfg, rembg_session, TIER)
        write_manifest(rec, manifest)
        print(f"[{i}/{len(images)}] {img.name[:40]} -> {rec['seconds']}s, "
              f"VRAM {rec['peak_vram_gb']}GB, {rec['glb_bytes']//1024}KB")
    except Exception as e:
        print(f'[{i}] GAGAL {img.name[:40]}: {e}')

In [ ]:
#@title 6. Zip hasil untuk diunduh (Output panel Kaggle)
import shutil
shutil.make_archive('/kaggle/working/glb_outputs', 'zip', '/kaggle/working/outputs')
print('Unduh: /kaggle/working/glb_outputs.zip (tab Output)')
!ls -la /kaggle/working/outputs/glb | head

## Catatan Kaggle
- Sesi GPU bisa putus → set `LIMIT` kecil dulu, naikkan bertahap. `outputs/` + `manifest.jsonl` jadi titik serah-terima.
- Untuk **commit hasil** (versi reproducible): Save Version → output tersimpan, bisa dijadikan dataset turunan untuk Jobdesk 3 & 4.
- Jika `INPUT` salah, cek sel 1 — slug dataset menentukan path `/kaggle/input/<slug>/`.